# Product, again
## test the two hypotheses:
## A is $N \times d$, B is $d \times m$
## 1. $PP(AB) \leq min( PP(A), PP(B) )$
## 2. $(PP(A) / d) * (PP(B) / m) \leq PP(AB) / m$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy.linalg import qr

import math
import pandas as pd

import random

In [2]:
import PatnaikPearson as pp
import cupy as cp

In [3]:
num_iterations = 100
size_scale = 500

xx = pp.pp_dim_AB_experiment(num_iterations, size_scale)

0 687 912 1277
0 3.000000000000001 1.1500000000000004
generate_data_manifold: using svd
generate_data_manifold: using svd
0 625.5191913841787 674.2454685166466 826.3618805828569
0 is PP(AB) =  625.5191913841787  leq min(PP(A), PP(B)) =  674.2454685166466
0 is nu/d (AB) =  0.48983491885996766  leq min( nu/d (A), nu/d (B)) =  0.6471118876921353
0 is nu/d (A) * nu/d (B) =  0.4784125634864862  leq  nu/d (AB) =  0.48983491885996766
1 646 566 1034
1 1.9500000000000006 4.400000000000001
generate_data_manifold: using svd
generate_data_manifold: using svd
1 293.60995234631025 294.5525009168397 563.5069636997719
1 is PP(AB) =  293.60995234631025  leq min(PP(A), PP(B)) =  294.5525009168397
1 is nu/d (AB) =  0.2839554664857933  leq min( nu/d (A), nu/d (B)) =  0.5204107790050171
1 is nu/d (A) * nu/d (B) =  0.28361228041948755  leq  nu/d (AB) =  0.2839554664857933
2 698 908 1107
2 4.400000000000001 4.450000000000001
generate_data_manifold: using svd
generate_data_manifold: using svd
2 684.9482126389

In [5]:

pp_dim_A_vals = xx["pp_dim_A_vals"]
pp_dim_B_vals = xx["pp_dim_B_vals"]
pp_dim_AB_vals = xx["pp_dim_AB_vals"]
min_pp_dim_A_pp_dim_B_vals = xx["min_pp_dim_A_pp_dim_B_vals"]
max_pp_dim_A_pp_dim_B_vals = xx["max_pp_dim_A_pp_dim_B_vals"]
nu_over_d_A_vals = xx["nu_over_d_A_vals"]
nu_over_d_B_vals = xx["nu_over_d_B_vals"]
nu_over_d_AB_vals = xx["nu_over_d_AB_vals"]
min_nu_over_d_A_nu_over_d_B_vals = xx["min_nu_over_d_A_nu_over_d_B_vals"]
max_nu_over_d_A_nu_over_d_B_vals = xx["max_nu_over_d_A_nu_over_d_B_vals"]
nu_over_d_A_times_nu_over_d_B_vals = xx["nu_over_d_A_times_nu_over_d_B_vals"]

In [6]:
df = pd.DataFrame(xx)

In [9]:
print(df.shape)
#print(df.head(5))

(100, 11)


In [10]:
this_file = "pp_dim_product_conjectures.csv"
df.to_csv(this_file, index=False)

In [ ]:

num_iterations = 20
size_scale = 500

these_alphas = np.arange(0.1, 5.15, 0.05) # 100


uniform_draws = True
use_pareto = True
use_uniform = False
use_cauchy = False
verbose = False
use_svd = True

pp_dim_A_vals = np.zeros(num_iterations)
pp_dim_B_vals = np.zeros(num_iterations)
pp_dim_AB_vals = np.zeros(num_iterations)
min_pp_dim_A_pp_dim_B_vals = np.zeros(num_iterations)
max_pp_dim_A_pp_dim_B_vals = np.zeros(num_iterations)

nu_over_d_A_vals = np.zeros(num_iterations)
nu_over_d_B_vals = np.zeros(num_iterations)
nu_over_d_AB_vals = np.zeros(num_iterations)

min_nu_over_d_A_nu_over_d_B_vals = np.zeros(num_iterations)
max_nu_over_d_A_nu_over_d_B_vals = np.zeros(num_iterations)
nu_over_d_A_times_nu_over_d_B_vals = np.zeros(num_iterations)

for i in range(num_iterations):

    N = int(size_scale * ( 1 + np.random.uniform(0,1)))
    d = N + int(size_scale * (np.random.uniform(0,1) - 0.5))
    m = d + int(size_scale * np.random.uniform(0,1))
    print(i, N, d, m)

    
    alpha_1 = random.choice(these_alphas)
    alpha_2 = random.choice(these_alphas)
    print(i, alpha_1, alpha_2)

    A = pp.generate_data_manifold(N, d, alpha_1, uniform_draws, use_pareto, use_uniform, use_cauchy, verbose, use_svd)
    B = pp.generate_data_manifold(d, m, alpha_2, uniform_draws, use_pareto, use_uniform, use_cauchy, verbose, use_svd)

    pp_dim_A = pp.calculate_PatnaikPearson_dim(A)
    dim_A = A.shape[1]
    nu_over_d_A = pp_dim_A / dim_A

    pp_dim_B = pp.calculate_PatnaikPearson_dim(B)
    dim_B = B.shape[1]
    nu_over_d_B = pp_dim_B / dim_B

    cp_AB = cp.matmul(cp.array(A), cp.array(B))
    AB = cp.asnumpy(cp_AB)
    pp_dim_AB = pp.calculate_PatnaikPearson_dim(AB)
    dim_AB = AB.shape[1]
    nu_over_d_AB = pp_dim_AB / dim_AB

    min_pp_dim_A_pp_dim_B = min(pp_dim_A, pp_dim_B)
    max_pp_dim_A_pp_dim_B = max(pp_dim_A, pp_dim_B)

    min_nu_over_d_A_nu_over_d_B = min(nu_over_d_A, nu_over_d_B)
    max_nu_over_d_A_nu_over_d_B = max(nu_over_d_A, nu_over_d_B)

    nu_over_d_A_times_nu_over_d_B = nu_over_d_A * nu_over_d_B

    pp_dim_A_vals[i] = pp_dim_A
    pp_dim_B_vals[i] = pp_dim_B
    pp_dim_AB_vals[i] = pp_dim_AB
    min_pp_dim_A_pp_dim_B_vals[i] = min_pp_dim_A_pp_dim_B
    max_pp_dim_A_pp_dim_B_vals[i] = max_pp_dim_A_pp_dim_B

    nu_over_d_A_vals[i] = nu_over_d_A
    nu_over_d_B_vals[i] = nu_over_d_B
    nu_over_d_AB_vals[i] = nu_over_d_AB

    min_nu_over_d_A_nu_over_d_B_vals[i] = min_nu_over_d_A_nu_over_d_B
    max_nu_over_d_A_nu_over_d_B_vals[i] = max_nu_over_d_A_nu_over_d_B
    nu_over_d_A_times_nu_over_d_B_vals[i] = nu_over_d_A * nu_over_d_B

    print(i, pp_dim_AB, pp_dim_A, pp_dim_B)
    print(i, "is PP(AB) = ", pp_dim_AB , " leq min(PP(A), PP(B)) = ", min(pp_dim_A, pp_dim_B))
    print(i, "is nu/d (AB) = ", nu_over_d_AB, " leq min( nu/d (A), nu/d (B)) = ", min(nu_over_d_A, nu_over_d_B)) 
    print(i, "is nu/d (A) * nu/d (B) = ", nu_over_d_A * nu_over_d_B, " leq  nu/d (AB) = ", nu_over_d_AB)


In [ ]:
this_title = "Pareto : PP(AB) <= min(PP(A),PP(B))\n A is N * d, B is d * m, for N,d,m between 500 and 1500"
plt.plot(pp_dim_AB_vals, pp_dim_AB_vals, linestyle = "--", color = "green", label = "PP(AB)")
plt.scatter(pp_dim_AB_vals, min_pp_dim_A_pp_dim_B_vals, label = "min(PP(A),PP(B))")
plt.scatter(pp_dim_AB_vals, max_pp_dim_A_pp_dim_B_vals, label = "max(PP(A),PP(B))")
plt.xlabel("PP(AB)")
plt.ylabel("PP")
plt.legend()
plt.title(this_title)
plt.savefig('pp_dim_AB_leq_min_pp_dim_A_pp_dim_B_pareto_random_dim.pdf', dpi=300, bbox_inches='tight')
#plt.savefig('nu_over_d_sigmoid_W_vs_nu_over_d_W_pareto_two.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
this_title = "Pareto : nu/d (AB) <= min(nu/d (A), nu/d (B))\n A is N * d, B is d * m, for N,d,m between 500 and 1500"

plt.plot(nu_over_d_AB_vals, nu_over_d_AB_vals, linestyle = "--", color = "green", label = "nu/d (AB)")
plt.scatter(nu_over_d_AB_vals, min_nu_over_d_A_nu_over_d_B_vals, label = "min(nu/d (A), nu/d (B))")
plt.scatter(nu_over_d_AB_vals, max_nu_over_d_A_nu_over_d_B_vals, label = "max(nu/d (A), nu/d (B))")
plt.xlabel("nu/d (AB)")
plt.ylabel("nu/d")
plt.legend()
plt.title(this_title)
plt.savefig('nu_over_d_AB_leq_min_nu_over_d_A_nu_over_d_B_pareto_random_dim.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
this_title = "Pareto : nu/d (A) * nu/d (B) <= nu/d (AB)\n A is N * d, B is d * m, for N,d,m between 500 and 1500"

plt.plot(nu_over_d_AB_vals, nu_over_d_AB_vals, linestyle = "--", color = "green", label = "nu/d (AB)")
plt.scatter(nu_over_d_AB_vals, nu_over_d_A_times_nu_over_d_B_vals, label = "nu/d (A) * nu/d (B))")
#plt.scatter(nu_over_d_AB_vals, max_nu_over_d_A_nu_over_d_B_vals, label = "max(nu/d (A), nu/d (B))")
plt.xlabel("nu/d (AB)")
plt.ylabel("nu/d")
plt.legend()
plt.title(this_title)
plt.savefig('nu_over_d_A_times_nu_over_d_B_leq_nu_over_d_AB_pareto_random_dim.pdf', dpi=300, bbox_inches='tight')
plt.show()